In [1]:
import duckdb
import pandas as pd
import os

In [2]:
con = duckdb.connect()

# Check if raw data exists
raw_path = "../data/raw/food.parquet"
if not os.path.exists(raw_path):
    raise FileNotFoundError(f"Raw data not found at {raw_path}")

query = """
SELECT DISTINCT
    f.code,
    f.brands,
    p.text AS product_name_en,
    i.text AS ingredients_text_en,
    f.ingredients,
    f.ingredients_tags,
    f.allergens_tags,
    f.traces_tags,
    f.categories,
    f.countries_tags
FROM '../data/raw/food.parquet' AS f
LEFT JOIN UNNEST(f.product_name) AS pn(p) ON p.lang = 'en'
LEFT JOIN UNNEST(f.ingredients_text) AS it(i) ON i.lang = 'en'
WHERE
    list_contains(f.countries_tags, 'en:philippines')
    AND f.ingredients IS NOT NULL
    AND i.text IS NOT NULL
"""

In [3]:
try:
    df = con.execute(query).fetchdf()
    print(f"Extracted {len(df)} rows")
except Exception as e:
    print(f"Query failed: {e}")
    raise

# Check empty ingredient texts (should be none)
empty_ingredients = (df['ingredients_text_en'].str.len() == 0).sum()
print(f"Rows with empty ingredients_text_en: {empty_ingredients}")

# Drop duplicates (already distinct, but safe)
df = df.drop_duplicates(subset=["code"])
print(f"After dropping duplicates: {len(df)} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extracted 1725 rows
Rows with empty ingredients_text_en: 0
After dropping duplicates: 1118 rows


In [4]:
# Ensure the interim directory exists
os.makedirs("../data/interim/", exist_ok=True)

# Save to interim
df.to_csv("../data/interim/extracted_raw.csv", index=False)
print("Saved to ../data/interim/extracted_raw.csv")

Saved to ../data/interim/extracted_raw.csv
